<a href="https://colab.research.google.com/github/AkankshaB123/ML/blob/main/Churn_Prediction_ANN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =============================================================================
#  Artificial Neural Network
# =============================================================================

# Installing Theano
# pip install --upgrade --no-deps git+git://github.com/Theano/Theano.git

# Installing Tensorflow
# Install Tensorflow from the website: https://www.tensorflow.org/versions/r0.12/get_started/os_setup.html

# Installing Keras
# pip install --upgrade keras

# Part 1 - Data Preprocessing


# =============================================================================
# Predict Customer Churn, whether the customer has churned (Exited) from Bank
# =============================================================================

# =============================================================================
# The independent variables will be
#Credit Score: reliability of the customer
#Geography: where is the customer from
#Gender: Male or Female
#Age
#Tenure: number of years of customer history in the company
#Balance: the money in the bank account
#Number of products of the customer in the bank
#Credit Card: if the customer has or not the CC
#Active: if the customer is active or not
#Estimated Salary: estimation of salary based on the entries
# =============================================================================

# =============================================================================
# Importing the libraries
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
import keras
from keras.models import Sequential
from keras.layers import Dense

In [4]:
# =============================================================================
# Importing the dataset
# =============================================================================
dataset = pd.read_csv('/content/sample_data/Churn_Modelling.csv')
dataset.head(5)

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [6]:
# =============================================================================
# Data Pre-Processing
# =============================================================================


#Creating the Dependent and Independent Features
X = dataset.iloc[:, 3:13].values
y = dataset.iloc[:, 13].values

In [8]:
# Encoding categorical data
labelencoder_X_1 = LabelEncoder()
X[:, 1] = labelencoder_X_1.fit_transform(X[:, 1])
labelencoder_X_2 = LabelEncoder()
X[:, 2] = labelencoder_X_2.fit_transform(X[:, 2])

# Fix for OneHotEncoder: `categorical_features` is deprecated.
# Use ColumnTransformer to apply OneHotEncoder to the 'Geography' column (index 1).
# `drop='first'` is used to handle the dummy variable trap.
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

ct = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'), [1]) # Apply OneHotEncoder to column 1, dropping the first category
    ],
    remainder='passthrough' # Keep other columns as they are
)

# Apply the transformation. ColumnTransformer returns a numpy array.
X = ct.fit_transform(X)

# The original code had `X = X[:, 1:]`. This was likely to drop one of the dummy variables
# for 'Geography' to avoid multicollinearity.
# Since `OneHotEncoder(drop='first')` already handles this, this line is no longer needed.
# If applied, it would incorrectly remove a valid feature after the transformation.

In [9]:
# Splitting the dataset into the Training set and Test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0)

# Feature Scaling for faster computation
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [11]:
# =============================================================================
# Building the Artifical Neural Network
# =============================================================================

# =============================================================================
# We can summarize 7 steps:
# 1. Randomly initialise the weights to small numbers close to 0, but not 0
# 2. Input the first observation of your dataset, in the input layer, each feature in one input node
# 3. Forward-Propagation: from left to right, the neurons are activated in a way the impact of each neuron’s activation is limited by the weights and it runs until getting the y
# 4. Compare the predicted result to the actual result. Measure the error generated
# 5. Back-propagation: from right to left. The error is back propagated. Update the weights according to how much they are responsible for the error. The learning rate decides how much we update the weights
# 6. Repeat steps 1 to 5 and update the weights after each observation (Reinforcement Learning). Repeat steps 1 to 5 but update the weights only after a batch of observation (Batch Learning)
# 7. When the whole training set passed through the ANN, that makes an epoch.
# =============================================================================


#nodes of hidden layers, based on experiments, is to choose the average between the input and the output layers.

# Initialising the ANN
classifier = Sequential()

In [15]:
# Adding the input layer and the first hidden layer
classifier.add(Dense(units = 6, kernel_initializer = 'uniform', activation = 'relu', input_dim = 11))
#Output_dim = nodes of first hidden layer (now 'units')
#init = initialization of the weights based on an uniform distribution (now 'kernel_initializer')
#activation = activation function
#input_dim = no. of input features

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [16]:
# Adding the second hidden layer
classifier.add(Dense(units = 6, kernel_initializer = 'uniform', activation = 'relu'))

# Adding the output layer
classifier.add(Dense(units = 1, kernel_initializer = 'uniform', activation = 'sigmoid'))

# Compiling the ANN
classifier.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

# Fitting the ANN to the Training set
classifier.fit(X_train, y_train, batch_size = 10, epochs = 100)

# Part 3 - Making the predictions and evaluating the model

# Predicting the Test set results
y_pred = classifier.predict(X_test)
y_pred = (y_pred > 0.5)

# Making the Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print(classification_report(y_pred,
            y_test))

Epoch 1/100
800/800 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.7961 - loss: 0.4726
Epoch 2/100
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7960 - loss: 0.4191
Epoch 3/100
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8194 - loss: 0.4060
Epoch 4/100
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8306 - loss: 0.3946
Epoch 5/100
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8299 - loss: 0.3857
Epoch 6/100
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8441 - loss: 0.3776
Epoch 7/100
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8481 - loss: 0.3726
Epoch 8/100
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8516 - loss: 0.3684
Epoch 9/100
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8491 - loss: 0.3659
Epoch 10/100
800/800 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8516 - loss: 0.3639
Epoch 11/100
800/800 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8521 - loss: 0.3632
Epoch 12/100
800/800 ━━━━━━━━━━━━━━━━━━━━